# Aufbau Wissensgraph mit RDF-Lib

In [2]:
from rdflib import Graph,URIRef,Literal,Namespace,FOAF,RDF,RDFS,XSD

import csv
import os
import pandas as pd
import kagglehub

path = kagglehub.dataset_download("nelgiriyewithana/world-stock-prices-daily-updating")

print("Path to dataset files:", path)
data_path = path+"\World-Stock-Prices-Dataset.csv"
data = pd.read_csv(data_path)

<>:11: SyntaxWarning: invalid escape sequence '\W'
<>:11: SyntaxWarning: invalid escape sequence '\W'
C:\Users\s3phi\AppData\Local\Temp\ipykernel_16436\2468708934.py:11: SyntaxWarning: invalid escape sequence '\W'
  data_path = path+"\World-Stock-Prices-Dataset.csv"
c:\workspace\StockPredictor\StockPredictor_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\s3phi\.cache\kagglehub\datasets\nelgiriyewithana\world-stock-prices-daily-updating\versions\389


In [10]:
brands = []
arr = data["Brand_Name"].unique()
for i in arr:
    brands.append(i.replace(" ", "-"))

brands

['peloton',
 'crocs',
 'the-coca-cola-company',
 'adidas',
 'american-express',
 'puma',
 'visa',
 'adobe',
 'unilever',
 'cisco',
 'jpmorgan-chase-&-co',
 'lvmh',
 'airbnb',
 'marriott',
 'zoominfo',
 'toyota',
 'hilton',
 "mcdonald's",
 'the-home-depot',
 'mastercard',
 'johnson-&-johnson',
 'uber',
 'procter-&-gamble',
 'coinbase',
 'fedex',
 '3m',
 'philips',
 'foot-locker',
 'netflix',
 'amd',
 'ubisoft',
 'shopify',
 'southwest-airlines',
 'logitech',
 'amazon',
 'apple',
 'nike',
 'target',
 'google',
 'spotify',
 'zoom-video-communications',
 'the-walt-disney-company',
 'roblox',
 'nintendo',
 'delta-air-lines',
 'microsoft',
 'costco',
 'american-eagle-outfitters',
 'colgate-palmolive',
 'pinterest',
 'bmw-group',
 'chipotle',
 'hershey-company',
 'porsche',
 'honda',
 'salesforce-/-slack',
 'nvidia',
 'starbucks',
 'tesla',
 'twitter',
 'nordstrom',
 'block']

In [11]:
#Liste mit Dictionaries: [{'stadt': 'Friedberg', 'bula': 'Hessen', 'einwohner': '62000'},...]

g = Graph() #Graph wird erzeugt -> Triple werden hinzugefügt

EX = Namespace("http://example.org/") #Definieren der Namensräume
EXC = Namespace("http://example.org/companies/")
g.bind("ex",EX) #In der Output-Datei (cities.ttl) wird der Namensraum mit dem Prefix verbunden
g.bind("exc",EXC)
g.bind("foaf",FOAF)

for brand in brands:
    #Mit g.add((...)) werden Triple zum Graph hinzugefügt.
    g.add((URIRef(EXC + brand),RDF.type,URIRef(EXC + "Company")))
    #URIRef = Es wird eine URI erzeugt. Der Namensraum RDF und die zugehörigen Properties
    #sind schon vordefiniert (Siehe RDF.type)
    #g.add((URIRef(EXC + brand),URIRef(EXC + "bula"),Literal(brand["bula"],lang="de"))) 
    #g.add((URIRef(EXC + brand),URIRef(EXC + "population"),Literal(brand["einwohner"],datatype=XSD.int)))
        #Mit Attribut datatype kann der Datentyp des Literals hinzugefügt werden.
        #Beispiel: Die Einwohnerzahl ist ein Integer.

    #Susanne und Marko leben in Friedberg und kennen sich




In [ ]:
g.parse()

In [18]:
q1 = """PREFIX dbo: <http://dbpedia.org/ontology/>
    SELECT ?plz
    WHERE {
      SERVICE <https://dbpedia.org/sparql> {
        <http://dbpedia.org/resource/Darmstadt> dbo:postalCode ?plz .
      }
    }
    LIMIT 1"""
    #q1 = Eine SPARQL-Anfrage an SPARQL-Endpoint dbpedia (SERVICE <URI des Endpoints>)
queryanswer = g.query(q1) #Anfrage q1 wird ausgeführt und liefert ein SPARQL-Object zurück.

print("queryanswer:",queryanswer)

for row in queryanswer:
    print(row)
    print(row[0])
    print(len(row))

queryanswer: <rdflib.plugins.sparql.processor.SPARQLResult object at 0x000002920E955880>
(rdflib.term.Literal('64283–64297'),)
64283–64297
1


In [ ]:

g.add((URIRef(EX + "Susanne"),URIRef(EXC + "lives_in"),URIRef(EXC + "Friedberg")))
g.add((URIRef(EX + "Marko"),URIRef(EXC + "lives_in"),URIRef(EXC + "Friedberg")))
g.add((URIRef(EX + "Susanne"),FOAF.knows,URIRef(EX + "Marko")))

    #PLZ Dieburg von dbpedia
q1 = """PREFIX dbo: <http://dbpedia.org/ontology/>
    SELECT ?plz
    WHERE {
      SERVICE <https://dbpedia.org/sparql> {
        <http://dbpedia.org/resource/Darmstadt> dbo:postalCode ?plz .
      }
    }
    LIMIT 1"""
    #q1 = Eine SPARQL-Anfrage an SPARQL-Endpoint dbpedia (SERVICE <URI des Endpoints>)

queryanswer = g.query(q1) #Anfrage q1 wird ausgeführt und liefert ein SPARQL-Object zurück.
    
for row in queryanswer:
    plz = str(row[0]) #row = (rdflib.term.Literal('64801–64807'),)
        #Es wird auf die erste Position zugegriffen und das Literal in einen Python-String umgewandelt.

    #g.add((URIRef(EXC + "Darmstadt"),URIRef(EXC + "plz"),Literal(plz)))

    #g.serialize("cities.ttl") #Der Graph wird in Turtle umgewandelt. Das nennt sich Serialisierung.

    #SPARQL-Abfrage mit rdflib
q2 = """SELECT ?city ?population WHERE {
	                  ?city a exc:City;
  	   		          exc:population ?population.
                      } ORDER BY ?population"""
#q2 = Eine SPARQL-Anfrage an Graph (g)

"""result = g.query(q2)
    
    for row in result:
        print(row)
        print("\t")""" #Enthält alle Städte und ihre Einwohnerzahlen.

In [25]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

In [47]:
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
# From https://www.wikidata.org/wiki/Wikidata:SPARQL_query_service/queries/examples#Cats
sparql.setQuery("""
SELECT ?officialname ?logo ?inception ?totalassets ?revenue ?netprofit ?operatingincome ?marketcapitalization
WHERE
{
  wd:Q26678 wdt:P1448 ?officialname;
            wdt:P154 ?logo;
            wdt:P571 ?inception;
            wdt:P2403 ?totalassets;
            wdt:P2139 ?revenue;
            wdt:P2295 ?netprofit;
            wdt:P3362 ?operatingincome;
            wdt:P2226 ?marketcapitalization.
}
""")
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

In [48]:
results

{'head': {'vars': ['officialname',
   'logo',
   'inception',
   'totalassets',
   'revenue',
   'netprofit',
   'operatingincome',
   'marketcapitalization']},
 'results': {'bindings': [{'officialname': {'xml:lang': 'de',
     'type': 'literal',
     'value': 'Bayerische Motoren Werke AG'},
    'logo': {'type': 'uri',
     'value': 'http://commons.wikimedia.org/wiki/Special:FilePath/BMW%20logo%20%28gray%29.svg'},
    'inception': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime',
     'type': 'literal',
     'value': '1916-03-07T00:00:00Z'},
    'totalassets': {'datatype': 'http://www.w3.org/2001/XMLSchema#decimal',
     'type': 'literal',
     'value': '228034000000'},
    'revenue': {'datatype': 'http://www.w3.org/2001/XMLSchema#decimal',
     'type': 'literal',
     'value': '142610000000'},
    'netprofit': {'datatype': 'http://www.w3.org/2001/XMLSchema#decimal',
     'type': 'literal',
     'value': '7680000000'},
    'operatingincome': {'datatype': 'http://www.w3.org/2001

In [62]:
print(results['results']['bindings'][0]["officialname"]["value"])
print(results['results']['bindings'][0]["inception"]["value"])
print(results['results']['bindings'][0]["totalassets"]["value"],"€")
print(results['results']['bindings'][0]["revenue"]["value"],"€")
print(results['results']['bindings'][0]["netprofit"]["value"],"€")
print(results['results']['bindings'][0]["operatingincome"]["value"],"€")
print(results['results']['bindings'][0]["marketcapitalization"]["value"],"€")
n = int(results['results']['bindings'][0]["marketcapitalization"]["value"])
res = "{:,}".format(n)
print(res,"€")

Bayerische Motoren Werke AG
1916-03-07T00:00:00Z
228034000000 €
142610000000 €
7680000000 €
13999000000 €
59907000000 €
59,907,000,000 €


In [50]:
results_df = pd.DataFrame(results['results']['bindings'])
results_df

,officialname,logo,inception,totalassets,revenue,netprofit,operatingincome,marketcapitalization
0,"{'xml:lang': 'de', 'type': 'literal', 'value':...","{'type': 'uri', 'value': 'http://commons.wikim...",{'datatype': 'http://www.w3.org/2001/XMLSchema...,{'datatype': 'http://www.w3.org/2001/XMLSchema...,{'datatype': 'http://www.w3.org/2001/XMLSchema...,{'datatype': 'http://www.w3.org/2001/XMLSchema...,{'datatype': 'http://www.w3.org/2001/XMLSchema...,{'datatype': 'http://www.w3.org/2001/XMLSchema...
